# 🧹 Notebook 02: Data Cleaning & Feature Extraction
**โครงการ**: Used Car Analytics (Data Warehouse & ETL Pipeline)

วัตถุประสงค์:
1. ทดสอบฟังก์ชัน `clean_price()` และ `clean_mileage()`
2. สกัด `model_year`, `brand`, `model`, `body_type` (Pick-up, SUV, Sedan, ฯลฯ), และ `fuel_type` ด้วย Regex
3. กรองข้อมูลขยะและค่า Null

In [1]:
import pandas as pd
import glob
import re

def standardize_scraped_columns(df):
    rename_map = {
        'data': 'car_title',
        'data2': 'description',
        'data3': 'mileage',
        'data4': 'location',
        'data6': 'car_model',
        'data16': 'transmission'
    }
    return df.rename(columns=rename_map)

raw_one2car_files = sorted(glob.glob('../../01_Raw_Data/one2car/one2car-11-*.csv'))
df_list = [standardize_scraped_columns(pd.read_csv(f)) for f in raw_one2car_files]
df_one2car = pd.concat(df_list, ignore_index=True)

print('One2car Raw Rows:', len(df_one2car))

One2car Raw Rows: 4190


In [2]:
def clean_price(val):
    if pd.isna(val): return None
    nums = re.sub(r'[^\d]', '', str(val))
    return float(nums) if nums != '' else None

def clean_mileage(val):
    if pd.isna(val): return None
    s = str(val).replace('กม.', '').replace(',', '').strip()
    match_range = re.search(r'(\d+)\s*-\s*(\d+)K', s, re.IGNORECASE)
    if match_range:
        low = float(match_range.group(1)) * 1000
        high = float(match_range.group(2)) * 1000
        return int((low + high) / 2)
    nums = re.sub(r'[^\d]', '', s)
    return int(nums) if nums != '' else None

def extract_body_type(title_str, desc_str=''):
    text = (str(title_str) + ' ' + str(desc_str)).lower()
    if any(k in text for k in ['pickup', 'cab', 'space cab', 'hi-lander', 'double cab', 'smart cab', 'revo', 'd-max', 'ranger', 'navara', 'กระบะ']):
        return 'Pick-up'
    elif any(k in text for k in ['suv', 'mu-x', 'fortuner', 'everest', 'cr-v', 'x3', 'glc', 'cross', 'hr-v', 'cx-5', 'pajero']):
        return 'SUV'
    elif any(k in text for k in ['hatchback', 'good cat', 'yaris', 'swift', '5 ประตู', 'ora']):
        return 'Hatchback'
    elif any(k in text for k in ['coupe', 'gran m sport', '220i']):
        return 'Coupe'
    elif any(k in text for k in ['van', 'caravelle', 'wagon', 'ตู้']):
        return 'Van'
    elif any(k in text for k in ['sedan', 'city', 'camry', 'altis', 'civic', 'mazda 3', 'c220', '520d', 'ซีดาน']):
        return 'Sedan'
    else:
        return 'Sedan'

def extract_fuel_type(title_str, desc_str=''):
    text = (str(title_str) + ' ' + str(desc_str)).lower()
    if any(k in text for k in ['e:hev', 'hev', 'hybrid', 'ไฮบริด']):
        return 'Hybrid'
    elif any(k in text for k in ['ora', 'good cat', 'ev', 'รถไฟฟ้า', '100%']):
        return 'EV'
    elif any(k in text for k in ['d-max', 'hilux', 'revo', 'ranger', 'navara', 'mu-x', 'fortuner', 'everest', '520d', 'c220 d', 'tdi', 'ดีเซล']):
        return 'Diesel'
    else:
        return 'Petrol'

def parse_car_title(title, desc=''):
    if pd.isna(title): return pd.Series([2018, 'Unknown', 'General', 'Sedan', 'Petrol'])
    title_str = str(title).strip()
    year_match = re.search(r'^(20\d{2}|19\d{2})', title_str)
    year = int(year_match.group(1)) if year_match else 2018
    text_clean = re.sub(r'^(20\d{2}|19\d{2})\s*', '', title_str)
    parts = text_clean.split()
    brand = parts[0] if len(parts) > 0 else 'Unknown'
    model = parts[1] if len(parts) > 1 else 'General'
    body_type = extract_body_type(title_str, desc)
    fuel_type = extract_fuel_type(title_str, desc)
    return pd.Series([year, brand, model, body_type, fuel_type])

# Apply Cleaning
df_clean = df_one2car.dropna(subset=['price']).copy()
df_clean['price_clean'] = df_clean['price'].apply(clean_price)
df_clean['mileage_clean'] = df_clean['mileage'].apply(clean_mileage)
df_clean[['model_year', 'brand', 'model', 'body_type', 'fuel_type']] = df_clean.apply(lambda r: parse_car_title(r.get('car_title'), r.get('description')), axis=1)
df_clean = df_clean.dropna(subset=['price_clean']).copy()

print('One2car Cleaned Rows:', len(df_clean))
display(df_clean[['car_title', 'brand', 'model', 'model_year', 'body_type', 'fuel_type', 'price_clean', 'mileage_clean']].head(5))

One2car Cleaned Rows: 3944


,car_title,brand,model,model_year,body_type,fuel_type,price_clean,mileage_clean
0,2015 Honda City 1.5 (ปี 14-18) SV+ Sedan - SV,Honda,City,2015,Sedan,Petrol,269000.0,172500
1,2023 Honda City 1.0 (ปี 19-26) SV Sedan,Honda,City,2023,Sedan,Petrol,359000.0,32500
2,2025 BMW 220i 2.0 F44 (ปี 20-27) Gran M Sport ...,BMW,220i,2025,Coupe,Petrol,1250000.0,32500
3,2022 Toyota HILUX REVO 2.4 Double Cab Z Editio...,Toyota,HILUX,2022,Pick-up,EV,449000.0,112500
4,2015 Honda City 1.5 (ปี 14-18) SV Sedan,Honda,City,2015,Sedan,Petrol,259000.0,22500
